# Chunking Strategies

## 2. Install the required packages

In [1]:
!uv pip install -q langchain-text-splitters pypdf openai tiktoken sentence-transformers matplotlib pandas python-dotenv

In [2]:
import os, re, json, statistics
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from pypdf import PdfReader

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

## 3. Load environment variables for API keys

In [3]:
load_dotenv()   # reads the .env file

if os.getenv("OPENAI_API_KEY"):
    print("API key found")
else:
    print("No API key found. Check your .env file (it is needed for transcription).")

API key found


## 4. Load the podcast audio file

In [4]:
AUDIO_PATH = Path("The_Blueprint_For_Trustworthy_AI.m4a")

print("Current working folder:", Path.cwd())
if AUDIO_PATH.exists():
    print(f"Audio found: {AUDIO_PATH.name} ({AUDIO_PATH.stat().st_size / 1_000_000:.1f} MB)")
else:
    print("Audio NOT FOUND. Move the file into the folder shown above.")

Current working folder: /Users/ben-macbook-air/Files/Projects/Ironhack Bootcamp/course work/week-3 /lab-chunking-strategies
Audio found: The_Blueprint_For_Trustworthy_AI.m4a (30.2 MB)


## 5. Transcribe the audio file (save it to a local file)

In [5]:
import subprocess, glob

TRANSCRIPT_PATH = Path("transcript.txt")

def transcribe_audio(audio_path, out_path, model="whisper-1", piece_seconds=600):
    """Transcribe audio with the OpenAI API and save the text. Skips all work if out_path already exists."""
    out = Path(out_path)
    if out.exists() and out.stat().st_size > 0:
        print(f"{out} already exists -> skipping transcription")
        return out.read_text(encoding="utf-8")

    # 1) shrink and cut the audio: mono, 64 kbps mp3, 10-minute pieces
    Path("_pieces").mkdir(exist_ok=True)
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(audio_path), "-ac", "1", "-b:a", "64k",
                    "-f", "segment", "-segment_time", str(piece_seconds), "-reset_timestamps", "1",
                    "_pieces/piece_%02d.mp3"], check=True)

    # 2) transcribe the pieces in order
    from openai import OpenAI
    client = OpenAI()                                  # uses OPENAI_API_KEY from the environment
    texts = []
    for f in sorted(glob.glob("_pieces/piece_*.mp3")):
        with open(f, "rb") as fh:
            texts.append(client.audio.transcriptions.create(model=model, file=fh).text)
        print(f, "done")

    text = " ".join(texts)
    out.write_text(text, encoding="utf-8")             # save to a local file
    return text

podcast_text = transcribe_audio(AUDIO_PATH, TRANSCRIPT_PATH)
print(f"Podcast transcript length: {len(podcast_text)} chars")
print(podcast_text[:200])

transcript.txt already exists -> skipping transcription
Podcast transcript length: 16461 chars
So imagine for a second you're driving across, I don't know, a massive suspension bridge. OK. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview t


## 6. Load the PDF document (from a URL or a local file)

In [6]:
PDF_PATH = Path("ai_hleg_ethics_guidelines_for_trustworthy_ai-en.pdf")

def load_pdf(path=PDF_PATH):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

pdf_text = load_pdf()
print(f"PDF length: {len(pdf_text)} chars")
print(pdf_text[:200])

PDF length: 156411 chars
 
 
 
 
 
 
INDEPENDENT 
HIGH-LEVEL EXPERT GROUP ON 
ARTIFICIAL INTELLIGENCE 
SET UP BY THE EUROPEAN COMMISSION 
 
 
 
 
 
 
ETHICS GUIDELINES 
FOR TRUSTWORTHY AI 
 
 
 
 
 
 
ETHICS GUIDELINES FOR TR


# Step 2 — Implement Fixed-Size Chunking

## 1. fixed-size chunks for both documents with `CharacterTextSplitter`.

In [7]:
splitter = CharacterTextSplitter(separator="", chunk_size=500, chunk_overlap=50)

splitter.split_text(podcast_text)

["So imagine for a second you're driving across, I don't know, a massive suspension bridge. OK. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview the welding crew. No. You just, you trust it. You just drive. You trust the bridge. You trust the engineering standards, the inspections, the laws that say this thing won't fail. Right. It's trust in the infrastructure. It's invisible, but it's there. Exactly. But now let's switch gears. Think about",
 "Exactly. But now let's switch gears. Think about the algorithm that just denied your mortgage application or the A.I. system scanning your face at the airport. Do you have that same trust? And I mean, honestly, should you? That is the defining question of our decade, I think. We've moved past the wow phase of A.I. and straight into the wait a minute phase. Today we are digging into the blueprint for that trust. We're unpacking a document that is arguably the Magna Carta for ethical computi

## Analysis questions

**1. Does fixed-size chunking break sentences in the middle?**

In [8]:
import statistics

def chunk_stats(chunks, label=""):
    sizes = [len(c) for c in chunks]
    clean_end = sum(1 for c in chunks if c.rstrip().endswith((".", "?", "!")))
    pct_clean = 100 * clean_end / len(chunks)
    print(f"{label}: n={len(chunks)}, avg={statistics.mean(sizes):.1f}, "
          f"min={min(sizes)}, max={max(sizes)}, clean_end={clean_end} ({pct_clean:.1f}%)")
    return sizes, clean_end, pct_clean

# Look at a boundary directly
splitter = CharacterTextSplitter(separator="", chunk_size=500, chunk_overlap=50)
podcast_chunks = splitter.split_text(podcast_text)

for i in (0, 1, 2):
    print(f"chunk {i} end:    ...{podcast_chunks[i][-50:]!r}")
    print(f"chunk {i+1} start: {podcast_chunks[i+1][:50]!r}...\n")

# Count how many chunks end "cleanly" (on sentence punctuation), varying chunk_size / overlap
print(f"{'size':>6} {'overlap':>8} {'n_chunks':>9} {'clean_end':>10} {'pct_clean':>10}")
for chunk_size in (500, 1000, 2000):
    for overlap in (0, 50, 100):
        splitter = CharacterTextSplitter(separator="", chunk_size=chunk_size, chunk_overlap=overlap)
        chunks = splitter.split_text(podcast_text)
        clean_end = sum(1 for c in chunks if c.rstrip().endswith((".", "?", "!")))
        pct_clean = 100 * clean_end / len(chunks)
        print(f"{chunk_size:6d} {overlap:8d} {len(chunks):9d} {clean_end:10d} {pct_clean:9.1f}%")


chunk 0 end:    ...". Exactly. But now let's switch gears. Think about"
chunk 1 start: "Exactly. But now let's switch gears. Think about t"...

chunk 1 end:    ...' arguably the Magna Carta for ethical computing, t'
chunk 2 start: 'arguably the Magna Carta for ethical computing, th'...

chunk 2 end:    ..."cally the Stone Age. We didn't have the generative"
chunk 3 start: "cally the Stone Age. We didn't have the generative"...

  size  overlap  n_chunks  clean_end  pct_clean
   500        0        33          3       9.1%
   500       50        37          2       5.4%
   500      100        41          3       7.3%
  1000        0        17          1       5.9%
  1000       50        18          2      11.1%
  1000      100        19          3      15.8%
  2000        0         9          1      11.1%
  2000       50         9          1      11.1%
  2000      100         9          2      22.2%


✍️ **Answer — Q1 (sentence breaks):**

Yes, fixed-size chunking breaks sentences in the middle almost every time. Looking at the printed
boundaries above, chunk 0 ends mid-sentence ("...switch gears. Think about") and chunk 1 starts
exactly where it was cut off ("Exactly. But now let's switch gears. Think about t..."), with no
regard for the sentence or word boundary.

Counting it: with `chunk_size=500, chunk_overlap=50` only **2 of 37** podcast chunks (5.4%) end
cleanly on `.`, `?` or `!`. The percentage changes with chunk size and overlap, but stays low
across the board — from **9.1%** (size 500, overlap 0) down to **5.4%** (size 500, overlap 50),
up to **22.2%** at the largest size/overlap combo tried (size 2000, overlap 100). Bigger chunks
and more overlap give slightly more chances to land on a sentence end by coincidence, but the
splitter never actually looks for sentence boundaries — it just counts characters.


**2. How does it handle paragraph boundaries?**

In [9]:
results = {}
for sep_name, sep in (("empty", ""), ("space", " "), ("default", "\n\n")):
    for label, text in (("podcast", podcast_text), ("pdf", pdf_text)):
        splitter = CharacterTextSplitter(separator=sep, chunk_size=500, chunk_overlap=50)
        chunks = splitter.split_text(text)
        sizes = [len(c) for c in chunks]
        over_limit = sum(1 for s in sizes if s > 500)
        results[(sep_name, label)] = (len(chunks), statistics.mean(sizes), min(sizes), max(sizes), over_limit)
        print(f"sep={sep_name:8s} text={label:8s} n={len(chunks):4d} "
              f"avg={statistics.mean(sizes):7.1f} min={min(sizes):5d} max={max(sizes):5d} "
              f"over_chunk_size={over_limit}")

print()
print("podcast '\\n\\n' count:", podcast_text.count("\n\n"))
print("pdf '\\n\\n' count:    ", pdf_text.count("\n\n"))


sep=empty    text=podcast  n=  37 avg=  493.1 min=  261 max=  500 over_chunk_size=0


Created a chunk of size 28061, which is longer than the specified 500


Created a chunk of size 29967, which is longer than the specified 500


Created a chunk of size 23694, which is longer than the specified 500


sep=empty    text=pdf      n= 348 avg=  498.0 min=  258 max=  500 over_chunk_size=0
sep=space    text=podcast  n=  37 avg=  490.8 min=  251 max=  500 over_chunk_size=0
sep=space    text=pdf      n= 342 avg=  495.4 min=  273 max=  500 over_chunk_size=0
sep=default  text=podcast  n=   1 avg=16461.0 min=16461 max=16461 over_chunk_size=1
sep=default  text=pdf      n=   4 avg=39093.0 min=23693 max=74680 over_chunk_size=4

podcast '\n\n' count: 0
pdf '\n\n' count:     3


In [10]:
bonus = {}
# A single separator can't be ".", "!" and "?" at once -- CharacterTextSplitter also
# accepts a REGEX as the separator (is_separator_regex=True). "[.!?\n]\s+" matches a
# period, question mark, exclamation mark, OR a line break, followed by whitespace --
# the "\n" is added so that headings/list items (which have no sentence punctuation
# at all) also count as a valid split point.
# keep_separator="end" keeps the matched punctuation+space attached to the end of
# each sentence (otherwise CharacterTextSplitter has a known quirk: with a regex
# separator it can literally re-insert the pattern text, or with a lookaround
# pattern it drops the separator and glues sentences back together with no space).
for label, text in (("podcast", podcast_text), ("pdf", pdf_text)):
    splitter = CharacterTextSplitter(
        separator=r"[.!?\n]\s+",
        is_separator_regex=True,
        keep_separator="end",
        chunk_size=500,
        chunk_overlap=50,
    )
    chunks = splitter.split_text(text)
    sizes = [len(c) for c in chunks]
    clean_end = sum(1 for c in chunks if c.endswith("\n") or c.rstrip().endswith((".", "?", "!")))
    over_limit = sum(1 for s in sizes if s > 500)
    bonus[label] = chunks
    print(f"text={label:8s} n={len(chunks):4d} avg={statistics.mean(sizes):7.1f} "
          f"min={min(sizes):5d} max={max(sizes):5d} clean_end={clean_end:4d} "
          f"({100*clean_end/len(chunks):.1f}%) over_chunk_size={over_limit}")

print()
for c in bonus["podcast"][:3]:
    print(repr(c[-60:]))


Created a chunk of size 602, which is longer than the specified 500


Created a chunk of size 546, which is longer than the specified 500


Created a chunk of size 518, which is longer than the specified 500


Created a chunk of size 1021, which is longer than the specified 500


Created a chunk of size 501, which is longer than the specified 500


...

Created a chunk of size 856, which is longer than the specified 500


Created a chunk of size 735, which is longer than the specified 500


Created a chunk of size 512, which is longer than the specified 500


Created a chunk of size 683, which is longer than the specified 500


text=podcast  n=  37 avg=  463.9 min=  339 max=  494 clean_end=  37 (100.0%) over_chunk_size=0
text=pdf      n= 385 avg=  404.9 min=  136 max=  959 clean_end= 357 (92.7%) over_chunk_size=9

"isible, but it's there. Exactly. But now let's switch gears."
'ase. Today we are digging into the blueprint for that trust.'
'yone listening, this document, it went public in April 2019.'


✍️ **Answer — Q2 (paragraph boundaries, plus comparing to a word separator):**

With the **default separator** (`"\n\n"`), `CharacterTextSplitter` only cuts where a double
newline exists — it never splits inside a "paragraph," even if that paragraph is far bigger than
`chunk_size`. The podcast transcript has **0** occurrences of `\n\n` (it's one continuous block
of speech-to-text), so it comes back as **1 single chunk of 16,461 characters** — 33x bigger than
the requested `chunk_size=500`. The PDF has only **3** occurrences of `\n\n`, so it becomes
**4 chunks**, with a max chunk of **74,680 characters** — again far above the 500-character
target. Both texts end up with **100% of chunks over `chunk_size`**, because the separator is
only a candidate split point — `chunk_size` is never enforced if that character never shows up.

`separator=""` and `separator=" "` behave almost identically here (37/342-348 chunks, all ≤500
chars): they ignore structure and just cut by raw count (or by count that happens to land on a
space), so they reliably respect the size limit but can slice through words or sentences freely.

A real sentence ends with `.`, `!` **or** `?` — one literal separator can't match all three.
`CharacterTextSplitter` also accepts a **regex** as the separator (`is_separator_regex=True`), so
`separator=r"[.!?]\s+"` matches *any* of them followed by whitespace. With `keep_separator="end"`
(to avoid two quirks: a literal regex pattern getting re-inserted as text, or a lookaround pattern
eating the space between sentences), this gave **100% clean sentence endings** on the podcast
(37/37) but only partially fixed the PDF — because a PDF also has **headings and list items that
end with a line break and no punctuation at all**, which `[.!?]\s+` can't see as a boundary.

Adding `\n` to the class — `separator=r"[.!?\n]\s+"` (see the cell below) — treats a plain line
break as a valid split point too. This changes the PDF picture noticeably: it goes from
**14 chunks over 500 chars (max 2,744)** with `[.!?]\s+` down to **9 chunks over 500 chars (max
959)** with `[.!?\n]\s+` — because headings/short lines that previously got glued onto a long
run of body text until the next `.`/`!`/`?` now get cut off right away. The "clean ending" rate
on the PDF correspondingly rises from a `.`/`!`/`?`-only count to **92.7%** once a line-break
ending also counts as "clean."

**Takeaway:** `chunk_size` with `CharacterTextSplitter` is a *soft cap* that only holds when the
separator is frequent enough in the text, **and** actually present at every kind of boundary the
text has. `""`/`" "` are frequent but ignore meaning; `"\n\n"` alone is far too rare; `[.!?]\s+`
respects sentence boundaries but misses structural ones like headings; adding `\n` closes most of
that gap for a structured document like the PDF. None of this is free of tradeoffs: the regex
approaches still make false splits on abbreviations ("A.I. powerful" → "...A.I." / "powerful..."),
and `chunk_size` is still only approximate — exactly the tradeoff `RecursiveCharacterTextSplitter`
is built to manage, by falling back through a list of separators instead of committing to one.


**3. Which content type handles fixed-size chunking better?**

In [11]:
for label, text in (("podcast", podcast_text), ("pdf", pdf_text)):
    splitter = CharacterTextSplitter(separator="", chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_text(text)
    chunk_stats(chunks, label)


podcast: n=37, avg=493.1, min=261, max=500, clean_end=2 (5.4%)


pdf: n=348, avg=498.0, min=258, max=500, clean_end=11 (3.2%)


✍️ **Answer — Q3 (podcast vs. PDF):**

Side by side (`chunk_size=500, chunk_overlap=50, separator=""`): the podcast transcript yields
**37 chunks** (avg 493.1, min 261, max 500) with **5.4%** ending cleanly on sentence punctuation,
while the PDF yields **348 chunks** (avg 498.0, min 258, max 500) with only **3.2%** ending
cleanly.

Neither text handles fixed-size chunking *well*, but the PDF is somewhat worse for clean
sentence boundaries. That matches what the raw text looks like: the podcast is a transcribed
conversation with normal sentence punctuation throughout, so a few splits land on `.`/`?`/`!`
just by chance. The PDF, in contrast, is extracted body text full of headings, bullet lists,
footnotes and page-break artifacts (and almost no `\n\n`, as shown in Q2) — so sentence-ending
punctuation is sparser relative to its length, and `separator=""` slices through headings,
list items and footnote numbers just as readily as through prose.

**Bonus — does a smarter, boundary-aware splitter change the picture?** Yes, and the PDF is
where it matters most. With `[.!?]\s+` (sentence punctuation only), the podcast hits 100% clean
endings and stays fully under the size cap, while the PDF reaches 100% clean endings too but
still has **14 of 381 chunks over 500 chars (max 2,744)** — because headings/list items have no
sentence punctuation to split on, so the splitter runs straight through them into the next
paragraph. Adding `\n` to the separator — `[.!?\n]\s+` — targets exactly that gap: it drops the
PDF's over-size chunks from 14 to **9** (max down to 959) and lifts its clean-ending rate to
**92.7%**, specifically because it can now also stop at a heading or a short line, not just at a
full sentence. The podcast's numbers don't change (still 37 chunks, 100% clean, 0 over the cap),
because it has no headings or list structure to begin with — it's just continuous speech.

So the real difference between the two content types isn't really about "sentences" — it's about
**document structure**. The podcast is prose from end to end, so a sentence-only separator already
chunks it predictably. The PDF mixes prose with structural elements (headings, lists, tables)
that don't follow sentence grammar at all, so it needs separators that understand *both* kinds of
boundary to chunk reliably — exactly the reasoning behind `RecursiveCharacterTextSplitter`'s
ordered list of separators (`["\n\n", "\n", " ", ""]`) instead of one fixed rule.


# Step 4 — Implement Token-Based Chunking

## 1. Token-based chunks for both documents with `TokenTextSplitter`.

In [12]:
import tiktoken

splitter = TokenTextSplitter(encoding_name="cl100k_base", chunk_size=300, chunk_overlap=50)
pdf_token_chunks = splitter.split_text(pdf_text)
podcast_token_chunks = splitter.split_text(podcast_text)
print(len(pdf_token_chunks), "PDF chunks,", len(podcast_token_chunks), "podcast chunks")

# Look at a boundary directly -- same check we did for CharacterTextSplitter in Step 2
enc = tiktoken.get_encoding("cl100k_base")
for i in (0, 1, 2):
    print(f"chunk {i} end:   ...{podcast_token_chunks[i][-50:]!r}")
    print(f"chunk {i+1} start: {podcast_token_chunks[i+1][:50]!r}...\n")


127 PDF chunks, 14 podcast chunks
chunk 0 end:   ...'is document, it went public in April 2019. In tech'
chunk 1 start: ' an independent group set up by the European Commi'...

chunk 1 end:   ...' of it like a three-legged stool. The first leg is'
chunk 2 start: '.I. as having three components. Right. And they ha'...

chunk 2 end:   ...' So lawful, ethical, and robust. The document then'
chunk 3 start: ' A.I., but if it crashes every time it sees a pixe'...



## Analysis questions

Same rule as Step 2: answer with **numbers from your own runs**, not "yes"/"no".

**1. Does token-based chunking also break words or sentences in the middle?**

In [13]:
# Count how many chunks end "cleanly" (on sentence punctuation), varying chunk_size / overlap
print(f"{'size':>6} {'overlap':>8} {'n_chunks':>9} {'clean_end':>10} {'pct_clean':>10}")
for chunk_size in (150, 300, 500):
    for overlap in (0, 50, 100):
        splitter = TokenTextSplitter(encoding_name="cl100k_base", chunk_size=chunk_size, chunk_overlap=overlap)
        chunks = splitter.split_text(podcast_text)
        clean_end = sum(1 for c in chunks if c.rstrip().endswith((".", "?", "!")))
        pct_clean = 100 * clean_end / len(chunks)
        print(f"{chunk_size:6d} {overlap:8d} {len(chunks):9d} {clean_end:10d} {pct_clean:9.1f}%")


  size  overlap  n_chunks  clean_end  pct_clean
   150        0        24          1       4.2%
   150       50        35          2       5.7%
   150      100        69          4       5.8%
   300        0        12          1       8.3%
   300       50        14          2      14.3%
   300      100        18          2      11.1%
   500        0         8          1      12.5%
   500       50         8          2      25.0%
   500      100         9          1      11.1%


✍️ **Answer — Q1 (word/sentence breaks):**

Yes. The boundary print above shows it directly: chunk 0 ends "...public in April 2019. In tech" and
chunk 1 starts " an independent group set up..." — a word ("tech[nology]") and a sentence cut mid-way,
exactly like `CharacterTextSplitter`.

Counting it: with `chunk_size=300, chunk_overlap=50`, only **2 of 14** podcast chunks (14.3%) end
cleanly on `.`, `?` or `!`. This is actually a bit *higher* than Step 2's character-based result at a
comparable size (5.4% at 500 chars), but it stays in the same ballpark and the same range as size/overlap
change — from **4.2%** (size 150, overlap 0) up to **25.0%** (size 500, overlap 50). The reason it isn't
zero is coincidence: `TokenTextSplitter` cuts by **token position**, with no concept of a sentence or
even a word — a token boundary landing right after a `.` token is luck, not design, same as a character
count landing on a `.` was luck in Step 2.

**Why it's not identical to character-based breaking:** tokens (BPE subword units) don't always split
on character runs the same way characters do — a word like "chunking" can be one token or two
("chunk" + "ing"), so a cut can fall *inside* a word at a point a character-count cut would never
naturally land on, or it can happen to land exactly on a word boundary because that word happened to
be a single token. Either way, the splitter has no sentence-awareness, so the structural problem from
Step 2 — chunk boundaries that ignore meaning — is identical in kind, even though the exact cut points
differ.


**2. How does token-based chunking compare to character-based chunking (Step 2)?**

> 💡 A token isn't a fixed number of characters — tokenizers use variable-length subword units.
> So "300 tokens" and "500 characters" aren't directly comparable sizes. To compare fairly, convert
> using the text's own **characters-per-token ratio**, then check: for the *same* target size, which
> splitter actually keeps the token count under control (the thing that matters for an LLM's context
> window), and which keeps the character count more consistent?

In [14]:
for label, text in (("podcast", podcast_text), ("pdf", pdf_text)):
    ratio = len(text) / len(enc.encode(text))  # chars per token, specific to this text

    tok_splitter = TokenTextSplitter(encoding_name="cl100k_base", chunk_size=300, chunk_overlap=50)
    tok_chunks = tok_splitter.split_text(text)

    # same target size, converted to characters, so the comparison is fair
    char_size = round(300 * ratio)
    char_overlap = round(50 * ratio)
    char_splitter = CharacterTextSplitter(separator="", chunk_size=char_size, chunk_overlap=char_overlap)
    char_chunks = char_splitter.split_text(text)

    tok_tokens = [len(enc.encode(c)) for c in tok_chunks]
    char_tokens = [len(enc.encode(c)) for c in char_chunks]
    tok_chars = [len(c) for c in tok_chunks]
    char_chars = [len(c) for c in char_chunks]

    print(f"{label}  (ratio={ratio:.2f} chars/token, char-equivalent target={char_size} chars)")
    print(f"  token-based:  n={len(tok_chunks):4d}  tokens[min,max]=({min(tok_tokens)},{max(tok_tokens)})   "
          f"chars[min,max]=({min(tok_chars)},{max(tok_chars)})")
    print(f"  char-based:   n={len(char_chunks):4d}  tokens[min,max]=({min(char_tokens)},{max(char_tokens)})   "
          f"chars[min,max]=({min(char_chars)},{max(char_chars)})")
    print()


podcast  (ratio=4.69 chars/token, char-equivalent target=1408 chars)
  token-based:  n=  14  tokens[min,max]=(257,300)   chars[min,max]=(1089,1503)
  char-based:   n=  14  tokens[min,max]=(281,322)   chars[min,max]=(1212,1408)



pdf  (ratio=4.93 chars/token, char-equivalent target=1480 chars)
  token-based:  n= 127  tokens[min,max]=(210,300)   chars[min,max]=(748,1724)
  char-based:   n= 127  tokens[min,max]=(256,425)   chars[min,max]=(1050,1480)



✍️ **Answer — Q2 (token-based vs. character-based):**

Converting fairly (300 tokens ≈ 1,408 characters for the podcast, ≈1,480 for the PDF, using each
text's own chars/token ratio) and comparing both splitters at that equivalent target:

- **Podcast:** token-based → 14 chunks, tokens range **257–300**, chars range 1,089–1,503.
  Char-based (same target) → 14 chunks, tokens range **281–322**, chars range 1,212–1,408.
- **PDF:** token-based → 127 chunks, tokens range **210–300**, chars range 748–1,724.
  Char-based (same target) → 127 chunks, tokens range **256–425**, chars range 1,050–1,480.

Both produce roughly the same **number** of chunks for the same target size — not surprising, since
both are mechanically "fixed window, slide forward." The real difference is **which unit stays under
control**: `TokenTextSplitter` guarantees every chunk is **at most 300 tokens** (that's the whole
point — it counts the actual unit an LLM's context window is measured in), while its character count
swings more (1,089–1,724 across both texts). `CharacterTextSplitter` is the mirror image: its
character count is tightly capped by `chunk_size`, but its **token count is not controlled at all** —
on the PDF it overshoots to **425 tokens**, 42% over the 300-token budget we were aiming for, because
some stretches of text use shorter, more token-dense wording than others.

**Takeaway:** if the downstream goal is "fit inside an LLM's context window" or "use roughly even API
cost per chunk," `TokenTextSplitter` is the right tool — it directly controls the unit that matters.
`CharacterTextSplitter` only controls a proxy for that unit, and the proxy can be off by more than 40%.


**3. Which content type handles token-based chunking better?**

In [15]:
for label, chunks in (("podcast", podcast_token_chunks), ("pdf", pdf_token_chunks)):
    sizes_chars = [len(c) for c in chunks]
    sizes_tokens = [len(enc.encode(c)) for c in chunks]
    clean_end = sum(1 for c in chunks if c.rstrip().endswith((".", "?", "!")))
    print(f"{label}: n={len(chunks)}, avg_chars={statistics.mean(sizes_chars):.1f}, "
          f"min_chars={min(sizes_chars)}, max_chars={max(sizes_chars)}, "
          f"avg_tokens={statistics.mean(sizes_tokens):.1f}, max_tokens={max(sizes_tokens)}, "
          f"clean_end={clean_end} ({100*clean_end/len(chunks):.1f}%)")


podcast: n=14, avg_chars=1386.9, min_chars=1089, max_chars=1503, avg_tokens=296.9, max_tokens=300, clean_end=2 (14.3%)


pdf: n=127, avg_chars=1477.9, min_chars=748, max_chars=1724, avg_tokens=299.3, max_tokens=300, clean_end=8 (6.3%)


✍️ **Answer — Q3 (podcast vs. PDF):**

At `chunk_size=300, chunk_overlap=50`: the podcast yields **14 chunks** (avg 1,386.9 chars /
296.9 tokens, max 300 tokens) with **14.3%** clean sentence endings. The PDF yields **127
chunks** (avg 1,477.9 chars / 299.3 tokens, max 300 tokens) with only **6.3%** clean endings —
the same ordering as Step 2's `CharacterTextSplitter` result (5.4% vs. 3.2%), for the same
underlying reason: the PDF has more headings, bullet lists and footnotes relative to its length,
so there's less sentence-ending punctuation per chunk for a token (or character) cut to land on
by chance.

What token-based chunking *does* fix, that character-based chunking never could: **both texts
stay almost exactly at the 300-token cap** (296.9 and 299.3 tokens on average, never over 300) —
compare that to Step 2, where `separator="\n\n"` let the PDF balloon to a 74,680-character chunk
with no limit at all. So for *size control*, token-based chunking handles both content types
equally well and reliably, regardless of their structure. For *boundary quality* (not cutting
mid-sentence), neither content type is handled "well" by `TokenTextSplitter` — it has no more
sentence-awareness than `CharacterTextSplitter`'s `separator=""` did, and the PDF is still the
harder text for that specific problem, same as in Step 2.
